# Lab 8A: Bitcoin as Financial Data

Audit the tape, estimate conditional risk, and test whether the result
travels

Bitcoin is a useful financial dataset because the market never closes,
the price lacks a cash-flow anchor, and the volatility regime has
changed repeatedly. Those features make a full-sample estimate easy to
obtain and hard to trust. This lab asks a narrower question than “is
crypto good or bad?”: what does this tape allow you to say about risk
tomorrow?

You will use the course Bloomberg parquet for FIN510. FIN306 can extract
the same Bitcoin series from Bloomberg Terminal in Semester 2 and then
run the same analysis. Report what this sample shows. Do not insert a
remembered volatility range or a current market statistic.

<figure>
<a
href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab07_crypto.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

In [ ]:
import sys
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf

try:
    from arch import arch_model
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "arch"])
    from arch import arch_model

for _root in (Path("scripts"), Path("../scripts"), Path("resources"), Path("../resources")):
    _candidate = _root.resolve()
    if _candidate.is_dir() and (_candidate / "bloomberg_loader.py").exists():
        sys.path.insert(0, str(_candidate))
        break

try:
    from bloomberg_loader import load_bloomberg
except ImportError:
    def load_bloomberg(tickers=None, **kwargs):
        raise FileNotFoundError("bloomberg_loader is not on the path")

sns.set_theme(style="whitegrid")

## Task 1: Audit the tape

Load `BTCUSD` with the shared loader. In Colab, the fallback reads the
course parquet published with the notebooks. It is the same course
object, not a live API request.

In [ ]:
try:
    market = load_bloomberg(tickers=["BTCUSD"])
except FileNotFoundError:
    url = (
        "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
        "main/resources/bloomberg_database.parquet"
    )
    market = pd.read_parquet(url)
    market = market[market["ticker"].eq("BTCUSD")]

btc = (
    market.loc[:, ["date", "ticker", "PX_LAST"]]
    .assign(date=lambda x: pd.to_datetime(x["date"]))
    .drop_duplicates(["ticker", "date"])
    .sort_values("date")
    .set_index("date")
)

print(btc.head())
print(btc.tail())
print("Rows:", len(btc))
print("Window:", btc.index.min().date(), "to", btc.index.max().date())
print("Missing prices:", int(btc["PX_LAST"].isna().sum()))
print("Duplicate dates:", int(btc.index.duplicated().sum()))

Before calculating a return, answer these questions:

-   Which ticker and price field are you using?
-   Does the file include weekends, and should a crypto series include
    them?
-   Is the last date a genuine end of the research window or merely the
    last export?
-   What cannot this consolidated price tell you about cross-exchange
    spreads or executable depth?

The last question matters. Makarov and Schoar (2020) find large
cross-border gaps during their study window. A single daily series is
suitable for a volatility exercise. It cannot recover the capital
controls, bank delays, or order-book depth that produced those gaps.

## Task 2: Returns and realised risk

Use log returns. Crypto trades every day, so the annualisation factor is
365 for this daily series. That convention scales the sample standard
deviation; it does not turn volatility into an observed annual outcome.

In [ ]:
btc["log_return"] = np.log(btc["PX_LAST"]).diff()
returns = btc["log_return"].dropna()

annualised_vol = returns.std() * np.sqrt(365)
summary = pd.Series({
    "daily_mean": returns.mean(),
    "daily_sd": returns.std(),
    "annualised_sd": annualised_vol,
    "skewness": returns.skew(),
    "excess_kurtosis": returns.kurt(),
    "one_percent_quantile": returns.quantile(0.01),
})
print(summary)

Plot the price, daily returns, and a 30-day rolling standard deviation.
Keep the rolling statistic in daily units first, then annualise only for
display.

In [ ]:
btc["rolling_vol_30d"] = returns.rolling(30).std() * np.sqrt(365)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axes[0].plot(btc.index, btc["PX_LAST"], color="black", linewidth=1)
axes[0].set_ylabel("BTCUSD")
axes[0].set_title("Price")

axes[1].plot(returns.index, 100 * returns, color="steelblue", linewidth=0.55)
axes[1].axhline(0, color="grey", linewidth=0.7)
axes[1].set_ylabel("Log return (%)")
axes[1].set_title("Daily return")

axes[2].plot(btc.index, 100 * btc["rolling_vol_30d"], color="darkred")
axes[2].set_ylabel("Annualised SD (%)")
axes[2].set_title("Thirty-day rolling volatility")
plt.tight_layout()
plt.show()

Write down the estimate you obtained, its date window, and the
annualisation convention. Then explain why “Bitcoin volatility is X%” is
incomplete without all three.

## Task 3: Distribution and memory

A return distribution can have little linear predictability in the mean
and strong predictability in its scale. Compare the autocorrelation of
returns with the autocorrelation of squared returns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_acf(returns, lags=30, ax=axes[0], title="ACF: Bitcoin returns")
plot_acf(returns**2, lags=30, ax=axes[1], title="ACF: squared Bitcoin returns")
plt.tight_layout()
plt.show()

jb = stats.jarque_bera(returns)
print("Jarque-Bera statistic:", jb.statistic)
print("Jarque-Bera p-value:", jb.pvalue)

Now inspect the tails without assuming that a rejection of normality
identifies the correct alternative distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(100 * returns, bins=60, density=True, color="slategrey", alpha=0.8)
axes[0].set_title("Daily log returns")
axes[0].set_xlabel("Return (%)")

stats.probplot(returns, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q plot")
plt.tight_layout()
plt.show()

Interpret the two ACF panels in financial terms. What can be forecast
more plausibly, tomorrow’s direction or tomorrow’s risk? Explain why a
small mean signal and persistent squared returns can coexist.

## Task 4: GARCH and GJR

Fit Student-$t$ GARCH(1,1) and GJR-GARCH(1,1) models. Percentage returns
keep the optimisation numerically well scaled.

In [ ]:
r_pct = 100 * returns

garch = arch_model(
    r_pct,
    mean="Constant",
    vol="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t",
).fit(disp="off")

gjr = arch_model(
    r_pct,
    mean="Constant",
    vol="GARCH",
    p=1,
    o=1,
    q=1,
    dist="t",
).fit(disp="off")

comparison = pd.DataFrame({
    "GARCH": {
        "alpha": garch.params["alpha[1]"],
        "beta": garch.params["beta[1]"],
        "gamma": np.nan,
        "AIC": garch.aic,
        "BIC": garch.bic,
    },
    "GJR": {
        "alpha": gjr.params["alpha[1]"],
        "beta": gjr.params["beta[1]"],
        "gamma": gjr.params["gamma[1]"],
        "AIC": gjr.aic,
        "BIC": gjr.bic,
    },
}).T

comparison["GARCH persistence"] = comparison["alpha"] + comparison["beta"]
print(comparison)

Plot both conditional-volatility paths against the returns.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(r_pct.index, r_pct, color="grey", linewidth=0.5)
axes[0].axhline(0, color="black", linewidth=0.6)
axes[0].set_ylabel("Return (%)")

axes[1].plot(garch.conditional_volatility, label="GARCH", linewidth=1.2)
axes[1].plot(gjr.conditional_volatility, label="GJR-GARCH", linewidth=1.0, alpha=0.8)
axes[1].set_ylabel("Conditional SD (%)")
axes[1].legend()
plt.tight_layout()
plt.show()

The sign and uncertainty of $\gamma$ decide whether this sample supports
an asymmetric response. Do not infer equity-style leverage merely
because the GJR model is available. Compare AIC and BIC, inspect the
parameter estimate, and state what the sample supports.

## Task 5: Walk-forward risk

A full-sample fit can describe the history and still fail as a forecast.
Reserve the final 90 observations. At each date, fit GARCH using only
the past and forecast the next conditional variance. Compare it with the
expanding historical mean of squared percentage returns.

In [ ]:
test_size = 90
start = len(r_pct) - test_size
records = []

for split in range(start, len(r_pct)):
    train = r_pct.iloc[:split]
    realised = r_pct.iloc[split] ** 2

    fitted = arch_model(
        train,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="t",
    ).fit(disp="off")

    garch_var = float(fitted.forecast(horizon=1).variance.iloc[-1, 0])
    mean_var = float((train - train.mean()).pow(2).mean())
    records.append({
        "date": r_pct.index[split],
        "realised_sq_return": realised,
        "garch_variance": garch_var,
        "mean_variance": mean_var,
    })

forecasts = pd.DataFrame(records).set_index("date")

Use both mean squared error and QLIKE. A squared return is a noisy proxy
for latent variance, so the loss function is part of the conclusion.

In [ ]:
def qlike(realised, forecast, eps=1e-8):
    forecast = np.maximum(np.asarray(forecast), eps)
    realised = np.maximum(np.asarray(realised), eps)
    return np.mean(np.log(forecast) + realised / forecast)

scores = pd.DataFrame({
    "MSE": [
        np.mean((forecasts["realised_sq_return"] - forecasts["garch_variance"]) ** 2),
        np.mean((forecasts["realised_sq_return"] - forecasts["mean_variance"]) ** 2),
    ],
    "QLIKE": [
        qlike(forecasts["realised_sq_return"], forecasts["garch_variance"]),
        qlike(forecasts["realised_sq_return"], forecasts["mean_variance"]),
    ],
}, index=["GARCH", "Expanding variance"])

print(scores)

Which forecast wins under each loss? If the rankings differ, explain why
that is an honest result rather than an inconvenience to hide.

## Task 6: Stability diagnostic

Split the sample at its midpoint and fit the same GARCH specification on
each half. This is a diagnostic for parameter instability. It is not, by
itself, a formal structural-break test.

In [ ]:
midpoint = len(r_pct) // 2
periods = {
    "Full sample": r_pct,
    "First half": r_pct.iloc[:midpoint],
    "Second half": r_pct.iloc[midpoint:],
}

rows = []
for label, sample in periods.items():
    fitted = arch_model(
        sample,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="t",
    ).fit(disp="off")
    alpha = fitted.params["alpha[1]"]
    beta = fitted.params["beta[1]"]
    rows.append({
        "period": label,
        "start": sample.index.min(),
        "end": sample.index.max(),
        "alpha": alpha,
        "beta": beta,
        "alpha_plus_beta": alpha + beta,
        "AIC_per_obs": fitted.aic / len(sample),
    })

stability = pd.DataFrame(rows).set_index("period")
print(stability)

A lower half-sample persistence estimate can be consistent with the full
sample mixing regimes. It does not prove that the full-sample
persistence is spurious. Sampling uncertainty, a changed shock
distribution, and a genuine break can all move the estimates. State
which explanation your diagnostics support and what further evidence
would distinguish them.

## Task 7: Investment memo

Write 300 words for an investment committee. Use your estimates rather
than a remembered crypto statistic. Cover four points:

-   the exact sample and data source;
-   the estimated annualised standard deviation and tail evidence;
-   whether GARCH improves the chosen walk-forward loss;
-   whether the stability diagnostic makes a single full-sample risk
    estimate credible.

Finish with one portfolio implication. A high-volatility asset can
dominate total risk even at a small capital weight. That is a
risk-budget statement, not a claim that the expected return is positive
or negative.

## What the lab establishes

The price tape can establish the realised return distribution in its own
window. It can show volatility clustering, compare conditional-variance
filters, and reveal whether a forecast beats a transparent benchmark. It
cannot establish that Bitcoin is a currency, that its users gained
welfare, or that a full-sample pattern was tradeable across fragmented
venues.

That boundary is professional statistical practice. Match the conclusion
to the data-generating process, preserve time in validation, and report
the loss under which one forecast beats another.

Makarov, Igor, and Antoinette Schoar. 2020. “Trading and Arbitrage in
Cryptocurrency Markets.” *Journal of Financial Economics* 135 (2):
293–319. <https://doi.org/10.1016/j.jfineco.2019.07.001>.